# 05 - Inference demo

Input: a list of mutations with VAF. Output: probability + binary call at the
saved operating threshold.

This is the cell to screen-record for a demo. Keep the caveat visible: the
output is a research risk score, not a diagnostic.

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.chdir("/content/drive/MyDrive/leukemia-prediagnose/blood-cancer-early-detection")
    # Run every session, not only on first mount: Colab wipes packages on
    # restart, and misspelled package names fail silently without check=True.
    subprocess.run("pip install -q transformers accelerate lightgbm".split(), check=True)
else:
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")

sys.path.insert(0, os.getcwd())
print("working dir:", os.getcwd())

import numpy as np, pandas as pd, yaml
import matplotlib.pyplot as plt
pd.set_option("display.width", 160)

CFG = yaml.safe_load(open("configs/config.yaml"))
CFG

In [ ]:
import json
from src.inference import Predictor

pred = Predictor(CFG["model"]["out_path"])
print(json.dumps(pred.meta, indent=2)[:700])

In [ ]:
# Dominant clone, spliceosome + TP53
case_a = [
    {"gene": "TP53",   "protein_change": "p.R248Q", "vaf": 0.31},
    {"gene": "SRSF2",  "protein_change": "p.P95H",  "vaf": 0.28},
    {"gene": "DNMT3A", "protein_change": "p.R882H", "vaf": 0.42},
]

# Single small clone
case_b = [
    {"gene": "DNMT3A", "protein_change": "p.R320C", "vaf": 0.04},
]

# A model trained with embeddings needs the embedder; load it once if so.
embedder = seqs = None
if pred.uses_embeddings:
    from src.embedding import ESM2Embedder
    from src.sequences import get_sequences
    print("model uses BioNeMo ESM-2 - loading embedder (~30s)...")
    embedder = ESM2Embedder(CFG["embedding"]["model_name"])
    seqs = get_sequences(sorted({m["gene"] for m in case_a + case_b}),
                         cache_path=CFG["data"]["uniprot_cache"])

for name, case in [("case A", case_a), ("case B", case_b)]:
    print(name, "->", pred.predict(case, embedder=embedder, sequences=seqs))

### If the model uses embeddings

`Predictor.predict` needs the embedder and sequence cache passed in. Loading the
650M model takes ~30s, so instantiate once and reuse.

In [ ]:
# Sanity: what the saved model actually is.
print("uses_embeddings:", pred.uses_embeddings)
print("threshold:", round(pred.threshold, 4))
print("genes in panel:", len(pred.genes))


In [ ]:
# Batch scoring from a CSV: patient_id,gene,protein_change,vaf
def score_csv(path):
    df = pd.read_csv(path)
    rows = []
    for pid, sub in df.groupby("patient_id"):
        r = pred.predict(sub[["gene", "protein_change", "vaf"]].to_dict("records"))
        r["patient_id"] = pid
        rows.append(r)
    return pd.DataFrame(rows)

print("ready - call score_csv('data/raw/new_samples.csv')")